<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            Agents | Foundations | xtra Lab 5
        </td>
        <td>
            <h2 style="color:#ff7800;">Exercise</h2>
            <span style="color:#ff7800;">Now try to build an Agent Loop from scratch yourself!<br/>
            Create a new .ipynb and make one from first principles, referring back to this as needed.<br/>
            It's one of the few times that I recommend typing from scratch - it's a very satisfying result.
            </span>
            <br><br>
            <span style="color: orange">Built from scratch, checklist is IEEE SRS<br/>
            </span>
        </td>
    </tr>
</table>

Let's make an Agent Loop with some familiar features borrowed from Claude Code:

1. A Terminal UI (TUI)
2. A Checklist tool to cause and track multiple tool calls

In [ ]:
from rich.console import Console
from dotenv import load_dotenv
from openai import OpenAI
import json
load_dotenv(override=True)

In [ ]:
def show(text):
    try:
        Console().print(text)
    except Exception:
        print(text)

In [ ]:
openai = OpenAI()

In [ ]:
checklist = []
completed = []

In [ ]:
def get_checklist_report() -> str:
    result = ""
    for index, item in enumerate(checklist):
        if completed[index]:
            result += f"Checklist #{index + 1}: [green][strike]{item}[/strike][/green]\n"
        else:
            result += f"Checklist #{index + 1}: {item}\n"
    show(result)
    return result

In [ ]:
get_checklist_report()

In [ ]:
def create_checklist(descriptions: list[str]) -> str:
    checklist.extend(descriptions)
    completed.extend([False] * len(descriptions))
    return get_checklist_report()

In [ ]:
def mark_complete(index: int, completion_notes: str) -> str:
    if 1 <= index <= len(checklist):
        completed[index -1] = True
    else:
        return "No checklist at this index."
    Console().print(completion_notes)
    return get_checklist_report()

In [ ]:
checklist, completed = [], []

create_checklist(["Introduction", "Overall Description", "Specific Requirements", "System Quality Requirements", "Verification and Traceability", "Final Review"])

In [ ]:
mark_complete(1, "complete")

In [ ]:
create_checklist_json = {
    "name": "create_checklist",
    "description": "Add new checklist from a list of description and return the full list",
    "parameters": {
        "type": "object",
        "properties": {
            "descriptions": {
                'type': 'array',
                'items': {'type': 'string'},
                'title': 'Descriptions of checklist items'
                }   
            },
        "required": ["descriptions"],
        "additionalProperties": False
    }
}

In [ ]:
mark_complete_json = {
    "name": "mark_complete",
    "description": "Mark complete the checklist item at the given position (starting from 1) and return the full list",
    "parameters": {
        'properties': {
            'index': {
                'description': 'The 1-based index of the checklist item to mark as complete',
                'title': 'Index',
                'type': 'integer'
                },
            'completion_notes': {
                'description': 'Notes about how you completed the checklist item in rich console markup',
                'title': 'Completion Notes',
                'type': 'string'  
                }
            },
        'required': ['index', 'completion_notes'],
        'type': 'object',
        'additionalProperties': False
    }
}

In [ ]:
tools = [{"type": "function", "function": create_checklist_json},
        {"type": "function", "function": mark_complete_json}]

In [ ]:
tools

In [ ]:
def handle_tool_calls(tool_calls):
    results = []
    for tool_call in tool_calls:
        tool_name = tool_call.function.name
        arguments = json.loads(tool_call.function.arguments)
        tool = globals().get(tool_name)
        result = tool(**arguments) if tool else {}
        results.append({"role": "tool","content": json.dumps(result),"tool_call_id": tool_call.id})
    return results

In [ ]:
def loop(messages):
    response = openai.chat.completions.create(model="gpt-5.5", messages=messages, tools=tools)
    while response.choices[0].finish_reason == "tool_calls":
        message = response.choices[0].message
        tool_calls = message.tool_calls
        results = handle_tool_calls(tool_calls)
        messages.append(message)
        messages.extend(results)
        response = openai.chat.completions.create(model="gpt-5.5", messages=messages, tools=tools)
    show(response.choices[0].message.content)

In [ ]:
system_message = """
You are given a problem to solve, by using your checklist tools to plan a list of steps, then carrying out each step in turn.
Now create a plan, set the checklist, carry out the steps, and reply with the solution. 
If any quantity isn't provided in the question, then include a step to come up with a reasonable estimate.
Provide your solution in Rich console markup without code blocks.
Do not ask the user questions or clarification; respond only with the answer after using your tools.
"""
user_message = """"
You are a Business Analyst and Systems Engineer. 
List a simple IEEE SRS checklist. 
Provide a time estimate for each item in the checklist for this Project. 
Provide summary estimates only. Do not provide detailed time estimate.

Project: Fictitious Autobiography Publishing Web Service
Assumptions: A simple web application where users register, write and edit their autobiographies, upload photos, preview content, and publish or share their autobiographies online. 
Estimates assume one person preparing the SRS, with basic stakeholder input and no complex integrations.
"""

Your task is to create a technical blueprint for building a simple Autobiography Publishing Web Service.
Using a simplified checklist based on the traditonal IEEE SRS, provide completion estimate. 

Autobiography Publishing helps customer turn their life story into a finished book, handling the editing, design, printing, 
and distribution. The customer keeps all rights to the book, choose who reads it, from family-only to the wider world.
Customer can update, correct, or remove content at anytime with Privacy and Security.
Project starts November 2026. Provide estimates. 

In [ ]:
checklist, completed = [], []
loop(messages)